# 01 — Game Design (Task 1)

**Group 7 — 6×6 Tic-Tac-Toe with Random Pre-allocation**

Goals for this notebook (Milestone 1):

1. Show the formal rules and the data structures used (`TicTacToe66`, `GameState`).
2. Implement / demonstrate the **randomised initial state** generator (1, 2, or 3 markers per player).
3. Visualise the board.
4. Sanity-check **win detection** (rows, columns, both diagonals, and the tie case).
5. Provide an interactive **human-vs-human** game loop to validate the rules.

All notebooks share configuration through `settings.py` and reusable code through `tictactoe66.py`.

In [1]:
import os, sys, random
# Make sure the project root is on sys.path so we can import settings/tictactoe66
_here = os.path.abspath(os.path.join(os.getcwd(), '..'))
if _here not in sys.path:
    sys.path.insert(0, _here)

import settings
import tictactoe66
from games4e import GameState

print('Board size :', settings.BOARD_SIZE, 'x', settings.BOARD_SIZE)
print('Win length :', settings.WIN_LENGTH)
print('Pre-alloc choices :', settings.PREALLOC_CHOICES)

Board size : 6 x 6
Win length : 4
Pre-alloc choices : (1, 2, 3)


## 1. The game class

`TicTacToe66` subclasses AIMA's `games4e.TicTacToe`, so it inherits `actions / result / utility / terminal_test` for free. We override `display` for a nicer ASCII board and add a cached `lines()` helper used by the heuristic in notebook 02.

In [2]:
game = tictactoe66.TicTacToe66()
print('All length-4 windows on the board:', len(game.lines()))
game.display(game.initial)

All length-4 windows on the board: 54
     1  2  3  4  5  6
   +------------------+
 1 |  .  .  .  .  .  . |
 2 |  .  .  .  .  .  . |
 3 |  .  .  .  .  .  . |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+


## 2. Random initial state

`make_initial_state(game, n_prealloc, rng)` places `n_prealloc` X's and `n_prealloc` O's uniformly at random in distinct empty cells, leaving X to move first.

In [3]:
rng = random.Random(settings.RANDOM_SEED)
for n in settings.PREALLOC_CHOICES:
    print(f'\n--- n_prealloc = {n} ---')
    s = tictactoe66.make_initial_state(game, n, rng=rng)
    game.display(s)
    print('to_move:', s.to_move, 'remaining moves:', len(s.moves))


--- n_prealloc = 1 ---
     1  2  3  4  5  6
   +------------------+
 1 |  .  O  .  .  .  . |
 2 |  .  X  .  .  .  . |
 3 |  .  .  .  .  .  . |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+
to_move: X remaining moves: 34

--- n_prealloc = 2 ---
     1  2  3  4  5  6
   +------------------+
 1 |  .  .  .  .  .  . |
 2 |  .  .  O  .  .  . |
 3 |  .  .  O  X  .  X |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+
to_move: X remaining moves: 32

--- n_prealloc = 3 ---
     1  2  3  4  5  6
   +------------------+
 1 |  O  .  O  .  .  X |
 2 |  X  .  .  .  .  . |
 3 |  .  .  .  .  .  . |
 4 |  .  .  .  .  .  . |
 5 |  .  .  .  O  .  . |
 6 |  .  .  .  .  X  . |
   +------------------+
to_move: X remaining moves: 30


## 3. Win-detection sanity checks

The base class's `compute_utility` already handles `k`-in-a-row. We verify that all four directions are detected, and that the tie path (full board, no winner) terminates.

In [4]:
def make_state(board, to_move='X'):
    moves = [(x, y) for x in range(1, game.h+1) for y in range(1, game.v+1) if (x, y) not in board]
    util = tictactoe66._scan_utility(game, board)
    return GameState(to_move=to_move, utility=util, board=board, moves=moves)

tests = {
    'row':       {(3, j): 'X' for j in (1,2,3,4)},
    'col':       {(i, 4): 'O' for i in (2,3,4,5)},
    'diag-down': {(i, i): 'X' for i in (1,2,3,4)},
    'diag-up':   {(5-i, 1+i): 'O' for i in (0,1,2,3)},
}
for name, b in tests.items():
    s = make_state(b)
    print(f'{name:10s}  utility={s.utility:+d}  terminal={game.terminal_test(s)}')

row         utility=+1  terminal=True
col         utility=-1  terminal=True
diag-down   utility=+1  terminal=True
diag-up     utility=-1  terminal=True


## 4. Human-vs-human game loop

Run the cell below to play a manual game and validate the rules end-to-end. Enter moves as `row,col` (1-indexed). Type `quit` to abort. *Skip this cell when running the notebook non-interactively.*

In [5]:
def human_player(game, state):
    while True:
        raw = input(f"{state.to_move} move (row,col): ").strip()
        if raw.lower() in ('q', 'quit'):
            raise SystemExit('aborted by user')
        try:
            r, c = (int(t) for t in raw.replace(' ', '').split(','))
        except ValueError:
            print('  bad input, try again, e.g. 3,4')
            continue
        if (r, c) in state.moves:
            return (r, c)
        print('  illegal, must be one of the empty cells')

# Interactive human-vs-human game.  Type "q" or "quit" at any prompt to abort.
res = tictactoe66.play_one_game(game, human_player, human_player,
                           n_prealloc=2, rng=random.Random(0), display=True)
print('result:', res)

     1  2  3  4  5  6
   +------------------+
 1 |  .  .  O  .  .  . |
 2 |  .  .  .  .  .  . |
 3 |  .  .  .  .  O  . |
 4 |  .  .  .  .  .  . |
 5 |  X  .  X  .  .  . |
 6 |  .  .  .  .  .  . |
   +------------------+


KeyboardInterrupt: Interrupted by user

**Milestone 1 reached:** game class, randomised initialisation, board visualisation, win detection, and a working human-vs-human loop. Notebook 02 layers AI agents on top.